# 3.1 Table One

**Class 3.** Almost every clinical research paper starts with a **"Table 1"**: a description of the people in the study, usually split by group, with a test of whether the groups differ. It tells readers **who** was studied and whether the groups are comparable. In this notebook we build one, comparing people **with** and **without diabetes**.

### By the end of this notebook you'll be able to

1. Tell **categorical** and **continuous** variables apart, and choose how to summarise each
2. Explain what a **p-value** is, and what a **chi-square test** and a **t-test** compare
3. Build a Table 1 in R by writing your own **functions**
4. Check a table for **small counts** before sharing it

| | |
|---|---|
| ⏱️ **Time** | 45–60 minutes |
| 📂 **Uses** | `final_analysis.csv` |
| 💾 **Saves** | nothing |

**Before you start:**

- Run **2.0 clean_and_join** (it saves `final_analysis.csv`)

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Key concepts

### Two kinds of variables

| Type | Values | Examples here | Summarise with | Compare groups with |
|---|---|---|---|---|
| **Categorical** | categories | gender, race, ethnicity, age group, PTSD answers | **count (%)** | **chi-square test** |
| **Continuous** | numbers on a scale | age, BMI, blood pressure, LDL, HbA1c | **mean (SD)**, or median (IQR) if skewed | **t-test** |

- **Mean (SD):** the average, and the **standard deviation**: how spread out the values are. About two thirds of values fall within one SD of the mean, if the data are roughly bell-shaped.
- **Median (IQR):** the middle value, and the **interquartile range** (the middle 50%). Better when data are **skewed** (a long tail of high values) or have outliers.

### Hypothesis tests and p-values

A test asks: **if the two groups were really the same, how surprising would a difference this big be?**

- We start by assuming **no difference** (the *null hypothesis*).
- The **p-value** is the probability of seeing a difference at least this large **if there were no real difference**.
- A small p-value (by convention **< 0.05**) means the difference would be surprising by chance alone, so we call it **statistically significant**.

What a p-value is **not**: the chance that the result is true, or a measure of how **big** or **important** a difference is. With large samples like All of Us, tiny, unimportant differences can have very small p-values. Always look at the **size** of the difference too.

### The two tests

- **Chi-square test** (categorical): compares the **counts** in each category with what we'd expect if group didn't matter. It needs reasonably large counts (a common rule: an expected count of at least 5 in each cell); for small tables, use **Fisher's exact test** instead.
- **t-test** (continuous): compares the **means** of two groups, taking their spread into account. R's `t.test()` uses **Welch's** version, which doesn't assume the two groups have the same SD.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

`prepare_analysis_data()` (in `aou_setup.R`) reads `final_analysis.csv` and turns text columns like `gender` into **factors**: R's type for categorical variables, with a fixed set of levels. For `diabetes`, `"No"` comes first, so it's treated as the reference group.

In [ ]:
fa <- prepare_analysis_data()
fa %>% count(diabetes)

## Writing your own functions

We need the same summary for many variables. Instead of copying code six times, we write a **function** once and run it for each variable:

```r
my_function <- function(input1, input2) {
  # steps that use input1 and input2
  result          # the last line is what the function returns
}
```

## Categorical variables: count (%) and chi-square

For one variable, `summarise_categorical()`:

1. **counts** people in each category by diabetes status with `table()`
2. runs a **chi-square test** on those counts with `chisq.test()`
3. formats each cell as **count (column %)** with `sprintf()`: `%d` is a whole number, `%.1f` a number with 1 decimal, `%%` a percent sign

`data[[var]]` picks the column whose **name** is stored in `var` (like `data$gender`, but the name can change). Percentages are **column percentages**: the share of each group in each category, so each group's percentages add up to 100%.

In [ ]:
summarise_categorical <- function(data, var) {
  tab <- table(data[[var]], data$diabetes)
  p <- suppressWarnings(chisq.test(tab)$p.value)
  as.data.frame.matrix(tab) %>%
    rownames_to_column("level") %>%
    mutate(
      `No diabetes`  = sprintf("%d (%.1f%%)", No,  100 * No  / sum(No)),
      `Diabetes`     = sprintf("%d (%.1f%%)", Yes, 100 * Yes / sum(Yes)),
      variable = var,
      p_value  = c(format.pval(p, digits = 2, eps = 0.001), rep("", n() - 1))
    ) %>%
    select(variable, level, `No diabetes`, `Diabetes`, p_value, n_no = No, n_yes = Yes)
}

categorical_vars <- c("gender", "race", "ethnicity", "age_group", "ptsd_doctor", "ptsd_treatment")
table_cat <- map_dfr(categorical_vars, ~ summarise_categorical(fa, .x))
table_cat %>% select(-n_no, -n_yes)

`map_dfr()` (from **purrr**) runs the function once for each variable in the list and **stacks** the results into one data frame. `~ summarise_categorical(fa, .x)` means "for each item, call it `.x`, and run this".

**🔍 Read the output:** for each variable, one row per category. The p-value appears on the first row of each variable. Compare the percentages across groups: where are they most different?

## Continuous variables: mean (SD) and t-test

In [ ]:
summarise_continuous <- function(data, var) {
  x_no  <- data[[var]][data$diabetes == "No"]
  x_yes <- data[[var]][data$diabetes == "Yes"]
  tibble(
    variable      = var,
    level         = "mean (SD)",
    `No diabetes` = sprintf("%.1f (%.1f)", mean(x_no),  sd(x_no)),
    `Diabetes`    = sprintf("%.1f (%.1f)", mean(x_yes), sd(x_yes)),
    p_value       = format.pval(t.test(x_yes, x_no)$p.value, digits = 2, eps = 0.001)
  )
}

continuous_vars <- c("age", "BMI", "Systolic_BP", "LDL_Cholesterol", "HbA1c")
table_cont <- map_dfr(continuous_vars, ~ summarise_continuous(fa, .x))
table_cont

`data[[var]][data$diabetes == "No"]` means "the values of this variable, for rows where diabetes is No". `format.pval()` writes p-values neatly: very small ones appear as `<0.001`.

## Put it together

In [ ]:
table_one <- bind_rows(
  tibble(variable = "N", level = "",
         `No diabetes` = as.character(sum(fa$diabetes == "No")),
         `Diabetes`    = as.character(sum(fa$diabetes == "Yes")), p_value = ""),
  table_cont,
  table_cat %>% select(-n_no, -n_yes)
)
table_one

**🔍 Read the output:** this is your Table 1. The first row gives the size of each group.

## Before you share: check for small counts

The All of Us **Data and Statistics Dissemination Policy** doesn't allow sharing counts of **1 to 20**, or anything they could be worked out from (such as a percentage of a small group, or a total minus the other categories). This lists any cell in the table that's too small:

In [ ]:
table_cat %>%
  filter(between(n_no, 1, 20) | between(n_yes, 1, 20)) %>%
  select(variable, level, n_no, n_yes)

**🔍 Read the output:** ideally, **no rows**. Any rows listed **can't be shared** as they are: combine small categories (e.g., group races together) or report them as *"≤ 20"*. `between(x, 1, 20)` is `TRUE` when `x` is from 1 to 20.

### ✍️ Your turn

BMI is often **skewed** (a long tail of high values). Report BMI as **median (IQR)** instead of mean (SD) for each group. (Hint: `median()` and `quantile(x, c(0.25, 0.75))`.)

<details>
<summary><b>Show answer</b></summary>

```r
fa %>%
  group_by(diabetes) %>%
  summarise(
    median = median(BMI),
    q1     = quantile(BMI, 0.25),
    q3     = quantile(BMI, 0.75)
  )
```

The **IQR** runs from the 25th percentile (`q1`) to the 75th (`q3`). For skewed data, compare groups with the **Wilcoxon rank-sum test** instead of a t-test: `wilcox.test(BMI ~ diabetes, data = fa)`.

</details>

In [ ]:
# Your code here

### ✍️ Your turn

Look at your Table 1:

1. Which variables differ the most between people with and without diabetes?
2. HbA1c differs enormously. Why is that expected, and why would it be a problem to treat HbA1c as a "risk factor" here?
3. A difference in LDL cholesterol might have a p-value below 0.05 but be only 2 mg/dL. Is that important?

<details>
<summary><b>Show answer</b></summary>

1. Usually **HbA1c**, **BMI**, and **blood pressure**, which are all closely tied to diabetes.
2. HbA1c is used to **diagnose** diabetes, so people with diabetes have high HbA1c almost by definition. It reflects the condition rather than predicting it: a circular comparison.
3. Probably not **clinically**. With thousands of people, very small differences become **statistically** significant. Judge the **size** of the difference, not just the p-value.

Also remember we ran **many** tests at once. With 11 tests at 0.05, about one in 20 could be "significant" by chance even if nothing were different (the **multiple comparisons** problem).

</details>

## ⚠️ Common errors

| Problem | Fix |
|---|---|
| *'final_analysis.csv' does not exist* | Run 2.0 first |
| *Chi-squared approximation may be incorrect* | Some expected counts are small; use `fisher.test()` for that variable |
| `NA` in a mean | Missing values; 2.0's `drop_na()` should prevent this |

## ✅ Key takeaways

- **Categorical** → count (%) and chi-square; **continuous** → mean (SD) or median (IQR) and t-test (or Wilcoxon)
- A **p-value** measures surprise under "no difference", not importance
- With big data, focus on the **size** of differences
- Check for **counts of 1 to 20** before sharing anything

> 💡 **Going further:** the packages `gtsummary` and `tableone` build publication-ready tables in one line, e.g. `gtsummary::tbl_summary(fa, by = diabetes)`. Install with `install.packages("gtsummary")`.

**Next:** **3.2 logistic_regression**.